# Redukcja nadmiarowości czujników linii produkcyjnej za pomocą PROC GVARCLUS

## Podsumowanie dla kierownictwa

Wielostrefowa linia produkcyjna strumieniuje dziesiątki skorelowanych odczytów z czujników, z których wiele niesie ten sam sygnał źródłowy. Ten notatnik wykorzystuje **PROC GVARCLUS** (grupowanie zmiennych) do pogrupowania 13 czujników procesowych w cztery rozłączne klastry, a następnie odczytuje wartości R-kwadrat każdego klastra, aby wybrać jeden reprezentatywny czujnik na klaster — zmniejszając zakres monitoringu SPC bez utraty informacji o procesie. Trzy z czterech klastrów mapują się czysto na fizyczne strefy (średnie R-kwadrat 0,92, 0,93 i 0,96); czwarty to klaster jednokanałowy, który procedura wydzieliła osobno — przyglądamy się mu, zamiast go pomijać.

## Źródła danych

Wszystkie dane są generowane wewnętrznie za pomocą `call streaminit(20260531)` i `rand()` — bez zewnętrznych ani sieciowych danych wejściowych.

| Zbiór danych | Wiersze | Kluczowe zmienne | Opis |
|---------|------|---------------|-------------|
| `process_sensors` | 100 | `zone1_a`–`zone1_c`, `zone2_a`–`zone2_c`, `zone3_a`, `zone3_b`, `temp_in`, `temp_out`, `pressure_a`, `pressure_b`, `vibration` | Syntetyczne odczyty z 3-strefowej linii produkcyjnej. Czujniki w obrębie strefy dzielą ukryty sterownik (wysoka korelacja); dodano kanały międzystrefowe (temperatury powiązane ze strefą 1, ciśnienia ze strefą 2, drgania ze strefą 3), aby stworzyć realistyczną nadmiarowość. Pętla kroku DATA żąda 300 cykli, ale to środowisko ewaluacyjne działa w trybie nielicencjonowanym i zachowuje pierwsze 100 obserwacji — wystarczająco, by czysto odzyskać strukturę klastrów. |
| `clusters` (OUT=) | 13 | `Variable`, `Cluster`, `RSq_Own` | Jeden wiersz na czujnik wejściowy: klaster, do którego został przypisany, oraz jego R-kwadrat względem komponentu tego klastra. Wytworzone przez instrukcję `OUTPUT OUT=`. |

# Redukcja nadmiarowości czujników linii produkcyjnej

Na oprzyrządowanej linii produkcyjnej powszechne jest rejestrowanie dziesiątek czujników mierzących nakładające się zjawiska fizyczne — wiele termopar w jednej strefie, nadmiarowe przetworniki ciśnienia, sondy drgań śledzące ten sam wał. Podawanie każdego kanału do karty kontrolnej lub modelu podrzędnego marnuje wysiłek monitoringu i zwiększa współliniowość.

**Grupowanie zmiennych** rozwiązuje ten problem bezpośrednio. `PROC GVARCLUS` dzieli zmienne numeryczne na *rozłączne* klastry, gdzie każdy klaster jest podsumowany pierwszym głównym komponentem swoich elementów. Czujniki, które poruszają się razem, lądują w tym samym klastrze; inżynier może następnie zachować jeden reprezentatywny kanał na klaster.

W tym notatniku:

1. Syntetyzujemy odczyty z 3-strefowej linii z celowo nadmiarowymi czujnikami (pętla żąda 300 cykli; tutaj zachowywanych jest 100).
2. Grupujemy 13 kanałów za pomocą `PROC GVARCLUS` z `MAXCLUSTERS=4`.
3. Przechwytujemy przypisania klastrów w zbiorze wyjściowym i je podsumowujemy.
4. Interpretujemy wartości R-kwadrat, aby wybrać jeden czujnik na klaster do bieżącego SPC.

## Krok 1 — Generowanie syntetycznych danych z czujników

Symulujemy trzy strefy procesowe, każda z ukrytym sterownikiem (`zone1_a`, `zone2_a`, `zone3_a`). Pozostałe kanały są zaszumionymi funkcjami liniowymi sterownika swojej strefy, więc są silnie skorelowane w obrębie strefy, ale niemal niezależne między strefami. Wiążemy również temperatury wlotu/wylotu ze strefą 1, a dwa przetworniki ciśnienia ze strefą 2, naśladując nadmiarowość międzyprzyrządową widoczną na rzeczywistych liniach. Stały ziarnik losowy zapewnia powtarzalność przebiegu. Pętla żąda 300 cykli; w trybie nielicencjonowanym silnik zachowuje pierwsze 100 obserwacji, co zgłasza poniższa NOTE.

In [1]:
DANE process_sensors;
    CALL streaminit(20260531);
    POWTÓRZ cycle = 1 TO 300;
        /* Strefa 1: ukryty sterownik plus dwie nadmiarowe sondy */
        zone1_a = rand("normal", 0, 1);
        zone1_b = 0.90*zone1_a + rand("normal", 0, 0.30);
        zone1_c = 0.85*zone1_a + rand("normal", 0, 0.35);

        /* Strefa 2: ukryty sterownik plus dwie nadmiarowe sondy */
        zone2_a = rand("normal", 0, 1);
        zone2_b = 0.88*zone2_a + rand("normal", 0, 0.30);
        zone2_c = 0.82*zone2_a + rand("normal", 0, 0.40);

        /* Strefa 3: ukryty sterownik plus jedna nadmiarowa sonda */
        zone3_a = rand("normal", 0, 1);
        zone3_b = 0.90*zone3_a + rand("normal", 0, 0.30);

        /* Kanały międzyprzyrządowe powiązane z istniejącymi sterownikami */
        temp_in    = 180 + 4.0*zone1_a + rand("normal", 0, 1.5);
        temp_out   = 178 + 4.0*zone1_a + rand("normal", 0, 1.6);
        pressure_a = 2.5 + 0.60*zone2_a + rand("normal", 0, 0.20);
        pressure_b = 2.5 + 0.58*zone2_a + rand("normal", 0, 0.22);
        vibration  = 0.40 + 0.30*zone3_a + rand("normal", 0, 0.10);
        WYJŚCIE;
    KONIEC;
    USUŃ cycle;
WYKONAJ;



NOTE: DATA process_sensors

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote process_sensors (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.06 seconds
  cpu   0.06 seconds


## Krok 2 — Grupowanie czujników

Wywołujemy `PROC GVARCLUS` na wszystkich 13 kanałach. Instrukcja `VAR` wymienia czujniki numeryczne do grupowania. Ograniczamy podział do czterech klastrów za pomocą `MAXCLUSTERS=4` (spodziewamy się mniej więcej jednego klastra na fizyczną strefę, z niewielkim marginesem). `ODS GRAPHICS ON` z `PLOTS=ALL` włącza dendrogram klastrów zmiennych; silnik zapisuje ten SVG do katalogu roboczego zamiast renderować go w linii, więc strukturę, którą odczytujemy poniżej, pochodzi z wydrukowanej tabeli **Przypisania klastrów zmiennych** oraz tabeli wartości własnych na klaster.

Instrukcja `OUTPUT OUT=` zapisuje przypisania zmiennych do klastrów — wraz z R-kwadrat każdej zmiennej względem własnego klastra — do zbioru, na którym można później programować.

In [2]:
ODS GRAPHICS ON;

PROCEDURA gvarclus DANE=process_sensors maxclusters=4 PLOTS=ALL;
    ZMIENNA zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
    WYJŚCIE out=clusters;
WYKONAJ;

ODS GRAPHICS OFF;



                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: ODS Graphics is ON (width=640px, height=480px, format=SVG).
NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.
NOTE: ODS Graphics is OFF.


## Krok 3 — Ponowne uruchomienie z opcją SUMMARY

Uruchomienie procedury po raz drugi z opcją `SUMMARY` zachowuje ten sam podział. `PLOTS=NONE` wyłącza grafikę w tym przebiegu. W tym środowisku wydrukowany raport jest identyczny jak w kroku 2 — ta sama 13-wierszowa tabela przypisań, te same cztery klastry i to samo podsumowanie wartości własnych — więc ta komórka głównie pokazuje, że opcje `SUMMARY` i `PLOTS=NONE` parsują się i działają poprawnie; nie oczekuje się nowych liczb.

In [3]:
PROCEDURA gvarclus DANE=process_sensors maxclusters=4 summary PLOTS=none;
    ZMIENNA zone1_a zone1_b zone1_c
        zone2_a zone2_b zone2_c
        zone3_a zone3_b
        temp_in temp_out
        pressure_a pressure_b
        vibration;
WYKONAJ;



                         The GVARCLUS Procedure

  Number of Variables            13
  Number of Observations         100
  Number of Clusters             4

Variable Cluster Assignments

  Variable                          Cluster    R-Squared
  --------                          -------    ---------
  zone1_a                                 1     0.968670
  zone1_b                                 1     0.918900
  zone1_c                                 4     1.000000
  zone2_a                                 2     0.983640
  zone2_b                                 2     0.946708
  zone2_c                                 2     0.892405
  zone3_a                                 3     0.977237
  zone3_b                                 3     0.949054
  temp_in                                 1     0.903394
  temp_out                                1     0.889519
  pressure_a                              2     0.929356
  pressure_b                              2     0.920915
  vibration  


NOTE: PROC GVARCLUS data=process_sensors

NOTE: Using Python sklearn version 1.8.0
NOTE: PROC GVARCLUS completed.


## Krok 4 — Analiza zbioru wyjściowego

Zbiór `clusters` z `OUTPUT OUT=` ma jeden wiersz na czujnik z przypisanym `Cluster` i `RSq_Own` (kwadratem korelacji między czujnikiem a komponentem jego klastra). Wysoki `RSq_Own` oznacza, że czujnik jest dobrze reprezentowany przez swój klaster — idealny kandydat do *pominięcia* na rzecz reprezentanta klastra. Sortujemy według klastra, a następnie według R-kwadrat, tak aby najbardziej reprezentatywny czujnik w każdym klastrze znalazł się na szczycie swojej grupy.

In [4]:
PROCEDURA SORTUJ DANE=clusters out=clusters_ranked;
    WEDŁUG CLUSTER MALEJĄCO RSq_Own;
WYKONAJ;

PROCEDURA DRUKUJ DANE=clusters_ranked ETYKIETA;
    ZMIENNA Variable CLUSTER RSq_Own;
    ETYKIETA Variable = "Kanał czujnika"
          CLUSTER  = "Klaster"
          RSq_Own  = "R-kwadrat z własnym klastrem";
WYKONAJ;



  Obs   Kanał czujnika  Klaster   R-kwadrat z własnym klastrem
-----  ---------------  -------  -----------------------------
    1  ZONE1_A                1                        0.96867
    2  ZONE1_B                1                         0.9189
    3  TEMP_IN                1                       0.903394
    4  TEMP_OUT               1                       0.889519
    5  ZONE2_A                2                        0.98364
    6  ZONE2_B                2                       0.946708
    7  PRESSURE_A             2                       0.929356
    8  PRESSURE_B             2                       0.920915
    9  ZONE2_C                2                       0.892405
   10  ZONE3_A                3                       0.977237
   11  VIBRATION              3                        0.95916
   12  ZONE3_B                3                       0.949054
   13  ZONE1_C                4                              1




NOTE: PROC SORT data=clusters

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 13 rows from clusters.
NOTE: Wrote clusters_ranked (13 rows, 3 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=clusters_ranked

NOTE: PROC PRINT completed: 13 observations printed, 3 variables


## Interpretacja wyników

Podział odzyskuje większość fizycznej struktury linii, z jednym uczciwym zastrzeżeniem:

- **Klaster 1** gromadzi `zone1_a` (R²=0,969), `zone1_b` (0,919) oraz temperatury wlotu/wylotu `temp_in` (0,903) i `temp_out` (0,890) — cztery z pięciu kanałów napędzanych sygnałem ukrytym strefy 1. Średnie R-kwadrat 0,920.
- **Klaster 2** gromadzi wszystkie trzy kanały strefy 2 `zone2_a` (0,984), `zone2_b` (0,947), `zone2_c` (0,892) razem z dwoma przetwornikami ciśnienia `pressure_a` (0,929) i `pressure_b` (0,921), które powiązaliśmy ze sterownikiem strefy 2. Średnie R-kwadrat 0,935.
- **Klaster 3** gromadzi `zone3_a` (0,977), `zone3_b` (0,949) i `vibration` (0,959). Średnie R-kwadrat 0,962.
- **Klaster 4** to pojedynczy kanał: `zone1_c`, trzecia sonda strefy 1, została wydzielona osobno z R²=1,000 (singleton jest, trywialnie, doskonale wyjaśniony samym sobą). Mimo współdzielenia sterownika strefy 1, przy tej wielkości próby procedura uznała `zone1_c` za wystarczająco odrębny od grupy `zone1_a`/temperatur, by zasłużyć na własny klaster, zamiast łączyć go z Klastrem 1.

Trzy wielokanałowe klastry niosą każdy średnie R-kwadrat powyżej **0,90**, potwierdzając, że jeden komponent wyjaśnia większość zmienności wśród swoich elementów — dokładnie tę nadmiarowość, którą chcemy zredukować. Samotny wyjątek — `zone1_c` lądujący w swoim własnym klastrze zamiast z resztą strefy 1 — jest użytecznym przypomnieniem, że grupowanie zmiennych jest sterowane danymi: przy większej liczbie cykli lub wyższym budżecie `MAXCLUSTERS` granica może się przesunąć, więc podział jest punktem wyjścia dla inżynierskiej oceny, a nie ustaloną prawdą.

**Działanie dla linii.** W obrębie każdego wielokanałowego klastra zachowaj czujnik o najwyższym `RSq_Own` (kanał najbardziej reprezentatywny dla swojego klastra), a resztę wycofaj lub obniż priorytet w rutynowym rejestrowaniu SPC — na przykład `zone2_a` dla Klastra 2 i `zone3_a` dla Klastra 3. Traktuj singleton `zone1_c` jako flagę do zbadania, a nie automatyczne zachowanie: potwierdź, czy niesie on rzeczywiście odrębną informację, czy jest artefaktem grupowania, zanim zdecydujesz się monitorować go osobno. Nawet z tym jednym kanałem odłożonym na bok, redukuje to 13 monitorowanych kanałów do planu monitoringu czterokanałowego, zmniejszając szum alarmów i współliniowość, przy zachowaniu treści informacyjnej linii.